# MicroGuard — A100 Full Pipeline
**Phase 1: Smoke Test → Phase 2: Train All → Phase 3: Max Quality Top 2**

**Runtime → Change runtime type → A100 GPU**

Run Cells 1-4 first (smoke test ~10 min). If all pass, run Cell 5 onwards.

In [ ]:
# ============================================================
# Cell 1: Setup + Mount Drive (~2 min)
# ============================================================
!pip install -q torch transformers datasets peft accelerate bitsandbytes
!pip install -q scikit-learn scipy pandas numpy tqdm evaluate

from google.colab import drive
drive.mount('/content/drive')

import os, gc, time, json, hashlib
import numpy as np
import torch
import torch.nn as nn
from datetime import datetime
from collections import Counter
from datasets import load_dataset, Dataset, DatasetDict, load_from_disk
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, f1_score, classification_report, cohen_kappa_score
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    TrainingArguments, Trainer, DataCollatorForSeq2Seq,
    RobertaForSequenceClassification, RobertaTokenizer,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

DRIVE_DIR = '/content/drive/MyDrive/MicroGuard_A100'
os.makedirs(f'{DRIVE_DIR}/results', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/models', exist_ok=True)
os.makedirs('results', exist_ok=True)
os.makedirs('models', exist_ok=True)

DEVICE = 'cuda'
SEED = 42
MAX_SEQ_LEN = 512

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print('Setup done!')


In [ ]:
# ============================================================
# Cell 2: Load or Download Data (~5 min first time, instant after)
# ============================================================
np.random.seed(SEED)

# Check all possible cache locations
data_cache = None
for path in [f'{DRIVE_DIR}/data_combined', '/content/drive/MyDrive/MicroGuard/data_combined', '/content/drive/MyDrive/MicroGuard_MaxQuality/data_combined']:
    if os.path.exists(f'{path}/dataset_info.json'):
        data_cache = path
        break

if data_cache:
    print(f'Loading from cache: {data_cache}')
    ds = load_from_disk(data_cache)
else:
    print('Downloading datasets (first time)...')
    RAGBENCH_SUBSETS = ['covidqa','cuad','delucionqa','emanual','expertqa','finqa','hagrid','hotpotqa','msmarco','pubmedqa','tatqa','techqa']
    ragbench_records = {'train':[],'validation':[],'test':[]}
    for subset in RAGBENCH_SUBSETS:
        try:
            print(f'  RAGBench/{subset}...')
            sub_ds = load_dataset('galileo-ai/ragbench', subset)
            for split in sub_ds:
                for ex in sub_ds[split]:
                    context = '\n\n'.join(ex['documents']) if isinstance(ex['documents'],list) else str(ex['documents'])
                    label = 'faithful' if ex['adherence_score'] else 'unfaithful'
                    ragbench_records[split].append({'query':ex['question'],'context':context,'answer':ex['response'],'label':label,'source':f'ragbench_{subset}'})
        except Exception as e:
            print(f'  Failed {subset}: {e}')

    ragtruth_records = {'train':[],'validation':[],'test':[]}
    try:
        print('  RAGTruth...')
        rt_ds = load_dataset('wandb/RAGTruth-processed')
        for split in rt_ds:
            for ex in rt_ds[split]:
                lp = ex['hallucination_labels_processed']
                if isinstance(lp,str): lp = json.loads(lp)
                has_h = lp.get('evident_conflict',0)>0 or lp.get('baseless_info',0)>0
                label = 'unfaithful' if has_h else 'faithful'
                ctx = ex.get('context') or ''; ans = ex.get('output') or ''
                if not ctx or not ans: continue
                t = 'test' if split=='test' else 'train'
                ragtruth_records[t].append({'query':ex.get('query',''),'context':ctx,'answer':ans,'label':label,'source':f"ragtruth_{ex.get('task_type','unknown')}"})
        if len(ragtruth_records['train'])>100:
            labels=[r['label'] for r in ragtruth_records['train']]
            ti,vi=train_test_split(range(len(ragtruth_records['train'])),test_size=0.1,stratify=labels,random_state=SEED)
            ragtruth_records['validation']=[ragtruth_records['train'][i] for i in vi]
            ragtruth_records['train']=[ragtruth_records['train'][i] for i in ti]
    except Exception as e:
        print(f'RAGTruth failed: {e}')

    halubench_records = {'train':[],'validation':[],'test':[]}
    try:
        print('  HaluBench...')
        hb_ds = load_dataset('PatronusAI/HaluBench')
        all_halu = []
        for ex in hb_ds['test']:
            label='faithful' if ex['label']=='PASS' else 'unfaithful'
            ctx=ex.get('passage',''); ans=str(ex.get('answer',''))
            if not ctx or not ans: continue
            all_halu.append({'query':ex.get('question',''),'context':ctx,'answer':ans,'label':label,'source':f"halubench_{ex.get('source_ds','unknown')}"})
        hl=[r['label'] for r in all_halu]
        tri,tei=train_test_split(range(len(all_halu)),test_size=0.2,stratify=hl,random_state=SEED)
        tel=[hl[i] for i in tei]
        vi2,ti2=train_test_split(tei,test_size=0.5,stratify=tel,random_state=SEED)
        halubench_records['train']=[all_halu[i] for i in tri]
        halubench_records['validation']=[all_halu[i] for i in vi2]
        halubench_records['test']=[all_halu[i] for i in ti2]
    except Exception as e:
        print(f'HaluBench failed: {e}')

    def dedup(recs):
        seen=set(); out=[]
        for r in recs:
            k=hashlib.md5((r['context'][:500]+r['answer'][:500]).encode()).hexdigest()
            if k not in seen: seen.add(k); out.append(r)
        return out
    combined={s:dedup(ragbench_records.get(s,[])+ragtruth_records.get(s,[])+halubench_records.get(s,[])) for s in ['train','validation','test']}
    ds=DatasetDict({s:Dataset.from_list(combined[s]) for s in combined})
    ds.save_to_disk(f'{DRIVE_DIR}/data_combined')

print(f'Dataset: Train={len(ds["train"])}, Val={len(ds["validation"])}, Test={len(ds["test"])}')
print(f'Labels: {dict(Counter(ds["train"]["label"]))}')


In [ ]:
# ============================================================
# Cell 3: Config + All Helper Functions
# ============================================================

SYSTEM_PROMPT = 'You are a faithfulness evaluator for RAG systems. You must respond with exactly one word.'
USER_TEMPLATE = """Context: {context}
Question: {query}
Answer: {answer}

Is every claim in the answer fully supported by the context? Respond with exactly one word: FAITHFUL or UNFAITHFUL."""

LORA_CONFIG = LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    bias='none',
)

ALL_MODELS = {
    'smollm135m':  {'name': 'HuggingFaceTB/SmolLM2-135M-Instruct', 'type': 'slm'},
    'smollm360m':  {'name': 'HuggingFaceTB/SmolLM2-360M-Instruct', 'type': 'slm'},
    'qwen05b':     {'name': 'Qwen/Qwen2.5-0.5B-Instruct', 'type': 'slm'},
    'gemma3_1b':   {'name': 'google/gemma-3-1b-it', 'type': 'slm'},
    'tinyllama':   {'name': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0', 'type': 'slm'},
}

ROBERTA_MODELS = {
    'roberta_base':  'roberta-base',
    'roberta_large': 'roberta-large',
}

def smart_truncate(q, c, a):
    return q[:200], c[:900], a[:400]

def format_prompt(q, c, a, label, tokenizer):
    q,c,a = smart_truncate(q,c,a)
    msg = USER_TEMPLATE.format(context=c, query=q, answer=a)
    target = 'FAITHFUL' if label=='faithful' else 'UNFAITHFUL'
    msgs = [{'role':'user','content':SYSTEM_PROMPT+'\n\n'+msg}, {'role':'assistant','content':target}]
    try: return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    except: return f'<|im_start|>user\n{SYSTEM_PROMPT}\n\n{msg}<|im_end|>\n<|im_start|>assistant\n{target}<|im_end|>'

def format_prompt_inference(q, c, a, tokenizer):
    q,c,a = smart_truncate(q,c,a)
    msg = USER_TEMPLATE.format(context=c, query=q, answer=a)
    msgs = [{'role':'user','content':SYSTEM_PROMPT+'\n\n'+msg}]
    try: return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    except: return f'<|im_start|>user\n{SYSTEM_PROMPT}\n\n{msg}<|im_end|>\n<|im_start|>assistant\n'

def tokenize_dataset(dataset, tokenizer):
    def fn(examples):
        ids,labels=[],[]
        for i in range(len(examples['query'])):
            ft=format_prompt(examples['query'][i],examples['context'][i],examples['answer'][i],examples['label'][i],tokenizer)
            pt=format_prompt_inference(examples['query'][i],examples['context'][i],examples['answer'][i],tokenizer)
            ftk=tokenizer(ft,truncation=True,max_length=MAX_SEQ_LEN,padding=False,return_tensors=None)
            ptk=tokenizer(pt,truncation=True,max_length=MAX_SEQ_LEN,padding=False,return_tensors=None)
            iid=ftk['input_ids']; pl=len(ptk['input_ids'])
            lb=[-100]*pl+iid[pl:]; lb=lb[:len(iid)]
            ids.append(iid); labels.append(lb)
        return {'input_ids':ids,'attention_mask':[[1]*len(i) for i in ids],'labels':labels}
    return dataset.map(fn,batched=True,batch_size=200,remove_columns=dataset.column_names,desc='Tokenizing')

def evaluate_model(model, tokenizer, dataset, device, max_samples=2000):
    """Constrained decoding evaluation — compare FAITHFUL vs UNFAITHFUL logits."""
    model.eval()
    yt,yp=[],[]
    indices=list(range(len(dataset)))
    if len(indices)>max_samples:
        np.random.seed(SEED)
        indices=np.random.choice(indices,max_samples,replace=False).tolist()
    faithful_ids = tokenizer.encode('FAITHFUL', add_special_tokens=False)
    unfaithful_ids = tokenizer.encode('UNFAITHFUL', add_special_tokens=False)
    t0=time.time()
    for i,idx in enumerate(indices):
        ex=dataset[idx]
        prompt=format_prompt_inference(ex['query'],ex['context'],ex['answer'],tokenizer)
        inp=tokenizer(prompt,return_tensors='pt',truncation=True,max_length=MAX_SEQ_LEN)
        inp={k:v.to(device) for k,v in inp.items()}
        with torch.no_grad():
            outputs = model(**inp)
            logits = outputs.logits[:,-1,:]
            f_score = logits[0, faithful_ids[0]].item()
            u_score = logits[0, unfaithful_ids[0]].item()
            pred = 'faithful' if f_score > u_score else 'unfaithful'
        yt.append(ex['label']); yp.append(pred)
        if (i+1)%500==0: print(f'    {i+1}/{len(indices)}...')
    elapsed=time.time()-t0
    ytb=[1 if l=='faithful' else 0 for l in yt]
    ypb=[1 if l=='faithful' else 0 for l in yp]
    ba=balanced_accuracy_score(ytb,ypb); f1=f1_score(ytb,ypb,average='macro')
    kp=cohen_kappa_score(ytb,ypb)
    print(f'    Bal.Acc={ba:.4f}, F1={f1:.4f}, Kappa={kp:.4f}, Time={elapsed:.0f}s')
    print(classification_report(yt,yp,digits=4))
    return {'balanced_accuracy':ba,'f1_macro':f1,'cohen_kappa':kp,'n_samples':len(yt),'n_skipped':0,'eval_time_seconds':elapsed,'per_sample_ms':elapsed/len(yt)*1000}

class FocalLossTrainer(Trainer):
    """Focal loss — lets model compute CE, then applies (1-pt)^gamma weighting."""
    def __init__(self, focal_gamma=2.0, **kwargs):
        super().__init__(**kwargs)
        self.focal_gamma = focal_gamma
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(**inputs)
        ce_loss = outputs.loss
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.focal_gamma) * ce_loss
        return (focal_loss, outputs) if return_outputs else focal_loss

def is_done(model_key, phase='phase1'):
    return os.path.exists(f'{DRIVE_DIR}/results/{phase}_{model_key}.json')

def save_results(results, model_key, phase='phase1'):
    path = f'results/{phase}_{model_key}.json'
    with open(path, 'w') as f: json.dump(results, f, indent=2)
    import shutil
    shutil.copy2(path, f'{DRIVE_DIR}/{path}')
    print(f'    Saved to Drive: {phase}_{model_key}.json')

def save_model_to_drive(model_key, phase='phase1'):
    import shutil
    src = f'models/{phase}_{model_key}/best'
    dst = f'{DRIVE_DIR}/models/{phase}_{model_key}/best'
    if os.path.exists(src):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        if os.path.exists(dst): shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f'    Model saved to Drive: {phase}_{model_key}')

def load_and_prepare_model(model_name):
    """Load model + tokenizer + apply LoRA. Returns (model, tokenizer)."""
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, trust_remote_code=True)
    try:
        model = get_peft_model(model, LORA_CONFIG)
    except ValueError:
        print('    LoRA fallback: q_proj, v_proj only')
        fallback = LoraConfig(task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32,
                              lora_dropout=0.05, target_modules=['q_proj','v_proj'], bias='none')
        model = get_peft_model(model, fallback)
    model.print_trainable_parameters()
    return model, tokenizer

def cleanup():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

print('All helpers ready!')


In [ ]:
# ============================================================
# Cell 4: 🔥 SMOKE TEST (~10 min)
# Tests every model end-to-end before committing to full training
# ============================================================
print('='*70)
print('SMOKE TEST — Testing all models before full training')
print('='*70)

# Subsample for smoke test
smoke_train = ds['train'].select(range(100))
smoke_test = ds['test'].select(range(50))

all_passed = True
smoke_results = {}

# Test SLMs
for model_key, config in ALL_MODELS.items():
    print(f'\n--- Testing {model_key} ({config["name"]}) ---')
    cleanup()
    try:
        # 1. Load model
        model, tokenizer = load_and_prepare_model(config['name'])
        print(f'  ✓ Model loaded')

        # 2. Tokenize
        tok_train = tokenize_dataset(smoke_train, tokenizer)
        print(f'  ✓ Tokenization works ({len(tok_train)} examples)')

        # 3. Train 5 steps
        collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, padding='max_length', max_length=MAX_SEQ_LEN, return_tensors='pt')
        args = TrainingArguments(
            output_dir=f'/tmp/smoke_{model_key}', max_steps=5,
            per_device_train_batch_size=4, gradient_accumulation_steps=1,
            learning_rate=2e-4, fp16=True, report_to='none',
            logging_steps=1, save_strategy='no',
            remove_unused_columns=False, seed=SEED,
        )
        trainer = Trainer(model=model, args=args, train_dataset=tok_train,
                          data_collator=collator, processing_class=tokenizer)
        result = trainer.train()
        print(f'  ✓ Training works (loss={result.training_loss:.4f})')

        # 4. Evaluate 50 examples
        metrics = evaluate_model(model, tokenizer, smoke_test, DEVICE, max_samples=50)
        print(f'  ✓ Evaluation works (bal_acc={metrics["balanced_accuracy"]:.4f})')

        # 5. Test focal loss trainer too
        focal_trainer = FocalLossTrainer(
            focal_gamma=2.0, model=model, args=args,
            train_dataset=tok_train, data_collator=collator, processing_class=tokenizer)
        focal_result = focal_trainer.train()
        print(f'  ✓ Focal loss works (loss={focal_result.training_loss:.4f})')

        mem = torch.cuda.max_memory_allocated()/1e9
        print(f'  ✓ Peak VRAM: {mem:.1f} GB')
        smoke_results[model_key] = 'PASS'

        del model, trainer, focal_trainer, tok_train
        cleanup()

    except Exception as e:
        print(f'  ✗ FAILED: {e}')
        import traceback; traceback.print_exc()
        smoke_results[model_key] = f'FAIL: {e}'
        all_passed = False
        try: del model, trainer
        except: pass
        cleanup()

# Test RoBERTa
for model_key, model_name in ROBERTA_MODELS.items():
    print(f'\n--- Testing {model_key} ({model_name}) ---')
    cleanup()
    try:
        tokenizer = RobertaTokenizer.from_pretrained(model_name)
        model = RobertaForSequenceClassification.from_pretrained(model_name, num_labels=2)
        # Quick tokenize test
        texts = [f"{smoke_train[i]['query'][:200]} </s> {smoke_train[i]['context'][:900]} </s> {smoke_train[i]['answer'][:400]}" for i in range(10)]
        tokens = tokenizer(texts, truncation=True, max_length=512, padding='max_length', return_tensors='pt')
        tokens['labels'] = torch.tensor([1 if smoke_train[i]['label']=='faithful' else 0 for i in range(10)])
        model.to(DEVICE)
        outputs = model(input_ids=tokens['input_ids'].to(DEVICE), attention_mask=tokens['attention_mask'].to(DEVICE), labels=tokens['labels'].to(DEVICE))
        print(f'  ✓ RoBERTa works (loss={outputs.loss.item():.4f})')
        smoke_results[model_key] = 'PASS'
        del model, tokenizer
        cleanup()
    except Exception as e:
        print(f'  ✗ FAILED: {e}')
        smoke_results[model_key] = f'FAIL: {e}'
        all_passed = False
        cleanup()

# Test Drive save
print(f'\n--- Testing Drive save ---')
try:
    test_file = f'{DRIVE_DIR}/results/smoke_test.json'
    with open(test_file, 'w') as f: json.dump(smoke_results, f)
    assert os.path.exists(test_file)
    print(f'  ✓ Drive save works')
except Exception as e:
    print(f'  ✗ Drive save FAILED: {e}')
    all_passed = False

print(f'\n{"="*70}')
print('SMOKE TEST RESULTS')
print(f'{"="*70}')
for k, v in smoke_results.items():
    status = '✓' if v == 'PASS' else '✗'
    print(f'  {status} {k}: {v}')

if all_passed:
    print(f'\n🎉 ALL TESTS PASSED — Safe to run Cell 5 (full training)!')
else:
    print(f'\n⚠️  SOME TESTS FAILED — Fix issues before running Cell 5!')


In [ ]:
# ============================================================
# Cell 5: PHASE 1 — Train ALL models with 40K data (~2.5 hours on A100)
# ============================================================
PHASE1_TRAIN_SAMPLES = 40000
NUM_EPOCHS = 3

# Subsample training data
train_labels = ds['train']['label']
keep_idx, _ = train_test_split(range(len(ds['train'])), train_size=PHASE1_TRAIN_SAMPLES, stratify=train_labels, random_state=SEED)
train_40k = ds['train'].select(sorted(keep_idx))
print(f'Phase 1 training set: {len(train_40k)} examples')

phase1_results = {}

# ---- Train SLMs ----
for model_key, config in ALL_MODELS.items():
    if is_done(model_key, 'phase1'):
        print(f'\n>>> SKIPPING {model_key} (already done!) <<<')
        with open(f'{DRIVE_DIR}/results/phase1_{model_key}.json') as f:
            phase1_results[model_key] = json.load(f)
        continue

    model_name = config['name']
    print(f'\n{"="*70}')
    print(f'Phase 1: {model_name} (40K data, standard CE)')
    print(f'{"="*70}')
    cleanup()

    try:
        model, tokenizer = load_and_prepare_model(model_name)
        train_ds = tokenize_dataset(train_40k, tokenizer)
        collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, padding='max_length', max_length=MAX_SEQ_LEN, return_tensors='pt')

        # A100 optimized: large batch, no grad checkpointing needed
        output_dir = f'models/phase1_{model_key}'
        args = TrainingArguments(
            output_dir=output_dir, num_train_epochs=NUM_EPOCHS,
            per_device_train_batch_size=32,
            gradient_accumulation_steps=1,
            learning_rate=2e-4, weight_decay=0.01, warmup_ratio=0.1,
            lr_scheduler_type='cosine', eval_strategy='no',
            save_strategy='epoch', save_total_limit=1,
            logging_steps=25, bf16=True, dataloader_num_workers=4,
            remove_unused_columns=False, report_to='none', seed=SEED,
            optim='adamw_torch_fused',
        )

        t0 = time.time()
        trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                          data_collator=collator, processing_class=tokenizer)
        train_result = trainer.train()
        training_time = time.time() - t0

        trainer.save_model(f'{output_dir}/best')
        tokenizer.save_pretrained(f'{output_dir}/best')

        print(f'  Evaluating...')
        metrics = evaluate_model(model, tokenizer, ds['test'], DEVICE)

        results = {
            'model': model_name, 'model_key': model_key, 'phase': 'phase1',
            'training_time_seconds': training_time,
            'training_time_formatted': f'{training_time/60:.1f}min',
            'peak_memory_mb': torch.cuda.max_memory_allocated()/1e6,
            'train_loss': train_result.training_loss,
            'train_samples': len(train_ds),
            'test_balanced_accuracy': metrics['balanced_accuracy'],
            'test_f1_macro': metrics['f1_macro'],
            'test_cohen_kappa': metrics['cohen_kappa'],
            'test_samples': metrics['n_samples'],
            'per_sample_latency_ms': metrics['per_sample_ms'],
            'timestamp': datetime.now().isoformat(),
        }
        save_results(results, model_key, 'phase1')
        save_model_to_drive(model_key, 'phase1')
        phase1_results[model_key] = results
        print(f'  *** {model_key}: Bal.Acc={metrics["balanced_accuracy"]:.4f}, F1={metrics["f1_macro"]:.4f}, Time={training_time/60:.1f}min ***')

        del model, trainer, train_ds
        cleanup()

    except Exception as e:
        print(f'  ERROR: {e}')
        import traceback; traceback.print_exc()
        phase1_results[model_key] = {'error': str(e)}
        try: del model, trainer
        except: pass
        cleanup()

# ---- Train RoBERTa baselines ----
def tokenize_roberta(dataset, tokenizer, max_len=512):
    def fn(examples):
        texts = [f"{q[:200]} </s> {c[:900]} </s> {a[:400]}" for q,c,a in zip(examples['query'],examples['context'],examples['answer'])]
        tokens = tokenizer(texts, truncation=True, max_length=max_len, padding='max_length')
        tokens['labels'] = [1 if l=='faithful' else 0 for l in examples['label']]
        return tokens
    return dataset.map(fn, batched=True, batch_size=200, remove_columns=dataset.column_names)

for model_key, model_name in ROBERTA_MODELS.items():
    if is_done(model_key, 'phase1'):
        print(f'\n>>> SKIPPING {model_key} (already done!) <<<')
        with open(f'{DRIVE_DIR}/results/phase1_{model_key}.json') as f:
            phase1_results[model_key] = json.load(f)
        continue

    print(f'\n{"="*70}')
    print(f'Phase 1: {model_name}')
    print(f'{"="*70}')
    cleanup()

    try:
        tokenizer = RobertaTokenizer.from_pretrained(model_name)
        model = RobertaForSequenceClassification.from_pretrained(model_name, num_labels=2)
        train_tok = tokenize_roberta(train_40k, tokenizer)
        val_tok = tokenize_roberta(ds['validation'], tokenizer)
        test_tok = tokenize_roberta(ds['test'], tokenizer)

        def compute_metrics(p):
            preds=p.predictions.argmax(-1)
            return {'balanced_accuracy':balanced_accuracy_score(p.label_ids,preds),'f1_macro':f1_score(p.label_ids,preds,average='macro')}

        args = TrainingArguments(
            output_dir=f'models/phase1_{model_key}', num_train_epochs=3,
            per_device_train_batch_size=64, per_device_eval_batch_size=128,
            learning_rate=2e-5, weight_decay=0.01, warmup_ratio=0.1,
            eval_strategy='epoch', save_strategy='epoch', save_total_limit=1,
            logging_steps=25, load_best_model_at_end=True,
            metric_for_best_model='eval_loss', bf16=True, report_to='none', seed=SEED,
        )

        t0=time.time()
        trainer=Trainer(model=model,args=args,train_dataset=train_tok,eval_dataset=val_tok,compute_metrics=compute_metrics)
        trainer.train()
        training_time=time.time()-t0

        out=trainer.predict(test_tok)
        preds=out.predictions.argmax(-1); labels=out.label_ids
        ba=balanced_accuracy_score(labels,preds); f1=f1_score(labels,preds,average='macro'); kp=cohen_kappa_score(labels,preds)
        yt=['faithful' if l==1 else 'unfaithful' for l in labels]
        yp=['faithful' if p==1 else 'unfaithful' for p in preds]
        print(classification_report(yt,yp,digits=4))

        results={'model':model_name,'model_key':model_key,'phase':'phase1',
                 'training_time_seconds':training_time,'train_loss':trainer.state.log_history[-1].get('train_loss',0),
                 'test_balanced_accuracy':ba,'test_f1_macro':f1,'test_cohen_kappa':kp,
                 'test_samples':len(labels),'timestamp':datetime.now().isoformat()}
        save_results(results, model_key, 'phase1')
        phase1_results[model_key] = results
        print(f'  *** {model_key}: Bal.Acc={ba:.4f}, F1={f1:.4f} ***')

        del model,trainer; cleanup()
    except Exception as e:
        print(f'  ERROR: {e}')
        import traceback; traceback.print_exc()
        phase1_results[model_key] = {'error': str(e)}
        cleanup()

# ---- NLI baseline ----
if not is_done('nli_deberta', 'phase1'):
    print(f'\n{"="*70}')
    print('Phase 1: NLI zero-shot baseline')
    print(f'{"="*70}')
    from transformers import pipeline
    nli = pipeline('zero-shot-classification', model='MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli', device=0)
    np.random.seed(SEED)
    nli_indices = np.random.choice(len(ds['test']), 2000, replace=False).tolist()
    yt,yp=[],[]
    t0=time.time()
    for i,idx in enumerate(nli_indices):
        ex=ds['test'][idx]
        try:
            r=nli(ex['answer'][:400],candidate_labels=['faithful','unfaithful'],hypothesis_template='This text is {}.')
            yt.append(ex['label']); yp.append(r['labels'][0])
        except: continue
        if (i+1)%500==0: print(f'    {i+1}/2000...')
    elapsed=time.time()-t0
    ytb=[1 if l=='faithful' else 0 for l in yt]; ypb=[1 if l=='faithful' else 0 for l in yp]
    ba=balanced_accuracy_score(ytb,ypb); f1=f1_score(ytb,ypb,average='macro'); kp=cohen_kappa_score(ytb,ypb)
    print(f'  NLI: Bal.Acc={ba:.4f}, F1={f1:.4f}')
    nli_results={'model':'DeBERTa-v3-NLI','model_key':'nli_deberta','phase':'phase1','method':'zero-shot',
                 'test_balanced_accuracy':ba,'test_f1_macro':f1,'test_cohen_kappa':kp,'n_samples':len(yt),
                 'eval_time_seconds':elapsed,'timestamp':datetime.now().isoformat()}
    save_results(nli_results, 'nli_deberta', 'phase1')
    phase1_results['nli_deberta'] = nli_results
    del nli; cleanup()

# ---- Phase 1 Summary ----
print(f'\n{"="*70}')
print('PHASE 1 RESULTS — ALL MODELS')
print(f'{"="*70}')
import pandas as pd
rows = []
for k, v in phase1_results.items():
    if 'error' not in v:
        rows.append({'Model': v.get('model',''), 'Bal.Acc': f"{v['test_balanced_accuracy']:.4f}",
                     'F1': f"{v['test_f1_macro']:.4f}", 'Time': v.get('training_time_formatted','N/A')})
if rows:
    print(pd.DataFrame(rows).to_string(index=False))

# Identify top 2 SLMs for Phase 2
slm_scores = {k:v['test_balanced_accuracy'] for k,v in phase1_results.items() if 'error' not in v and k in ALL_MODELS}
top2 = sorted(slm_scores, key=slm_scores.get, reverse=True)[:2]
print(f'\nTop 2 SLMs for Phase 2 (max quality): {top2}')
print(f'  {top2[0]}: {slm_scores[top2[0]]:.4f}')
print(f'  {top2[1]}: {slm_scores[top2[1]]:.4f}')


In [ ]:
# ============================================================
# Cell 6: PHASE 2 — Max Quality on Top 2 Models (~2.5 hours on A100)
# Full 98K data + Focal Loss
# ============================================================
print(f'\nPhase 2: Max quality training on top 2 models')
print(f'Models: {top2}')
print(f'Training data: FULL {len(ds["train"])} examples')
print(f'Loss: Focal (gamma=2.0)')

phase2_results = {}

for model_key in top2:
    if is_done(model_key, 'phase2'):
        print(f'\n>>> SKIPPING {model_key} (already done!) <<<')
        with open(f'{DRIVE_DIR}/results/phase2_{model_key}.json') as f:
            phase2_results[model_key] = json.load(f)
        continue

    config = ALL_MODELS[model_key]
    model_name = config['name']
    print(f'\n{"="*70}')
    print(f'Phase 2 MAX QUALITY: {model_name}')
    print(f'  Full 98K data + Focal Loss + 4 LoRA targets')
    print(f'{"="*70}')
    cleanup()

    try:
        model, tokenizer = load_and_prepare_model(model_name)
        print(f'  Tokenizing full dataset...')
        train_ds = tokenize_dataset(ds['train'], tokenizer)
        collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, padding='max_length', max_length=MAX_SEQ_LEN, return_tensors='pt')

        output_dir = f'{DRIVE_DIR}/models/phase2_{model_key}'  # Save to Drive
        args = TrainingArguments(
            output_dir=output_dir, num_train_epochs=3,
            per_device_train_batch_size=16,
            gradient_accumulation_steps=2,
            learning_rate=2e-4, weight_decay=0.01, warmup_ratio=0.1,
            lr_scheduler_type='cosine', eval_strategy='no',
            save_strategy='steps', save_steps=500, save_total_limit=2,
            logging_steps=25, bf16=True, dataloader_num_workers=4,
            remove_unused_columns=False, report_to='none', seed=SEED,
            optim='adamw_torch_fused',
        )

        # Resume from checkpoint if exists
        last_checkpoint = None
        if os.path.exists(output_dir):
            checkpoints = [d for d in os.listdir(output_dir) if d.startswith('checkpoint-')]
            if checkpoints:
                last_checkpoint = os.path.join(output_dir, sorted(checkpoints, key=lambda x: int(x.split('-')[1]))[-1])
                print(f'  Resuming from: {last_checkpoint}')

        t0 = time.time()
        trainer = FocalLossTrainer(
            focal_gamma=2.0, model=model, args=args,
            train_dataset=train_ds, data_collator=collator, processing_class=tokenizer)
        train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
        training_time = time.time() - t0

        trainer.save_model(f'{output_dir}/best')
        tokenizer.save_pretrained(f'{output_dir}/best')

        print(f'  Evaluating...')
        metrics = evaluate_model(model, tokenizer, ds['test'], DEVICE)

        results = {
            'model': model_name, 'model_key': model_key, 'phase': 'phase2_maxquality',
            'training_time_seconds': training_time,
            'training_time_formatted': f'{training_time/60:.1f}min',
            'peak_memory_mb': torch.cuda.max_memory_allocated()/1e6,
            'train_loss': train_result.training_loss,
            'train_samples': len(train_ds),
            'test_balanced_accuracy': metrics['balanced_accuracy'],
            'test_f1_macro': metrics['f1_macro'],
            'test_cohen_kappa': metrics['cohen_kappa'],
            'test_samples': metrics['n_samples'],
            'per_sample_latency_ms': metrics['per_sample_ms'],
            'config': 'full_98K+focal_loss+4lora+constrained_decode',
            'timestamp': datetime.now().isoformat(),
        }
        save_results(results, model_key, 'phase2')
        save_model_to_drive(model_key, 'phase2')
        phase2_results[model_key] = results
        print(f'  *** {model_key} MAX QUALITY: Bal.Acc={metrics["balanced_accuracy"]:.4f}, F1={metrics["f1_macro"]:.4f} ***')

        del model, trainer, train_ds
        cleanup()

    except Exception as e:
        print(f'  ERROR: {e}')
        import traceback; traceback.print_exc()
        phase2_results[model_key] = {'error': str(e)}
        try: del model, trainer
        except: pass
        cleanup()

print(f'\n{"="*70}')
print('PHASE 2 RESULTS')
print(f'{"="*70}')
for k,v in phase2_results.items():
    if 'error' not in v:
        print(f'  {k}: Bal.Acc={v["test_balanced_accuracy"]:.4f}, F1={v["test_f1_macro"]:.4f}')


In [ ]:
# ============================================================
# Cell 7: FINAL SUMMARY — All Results
# ============================================================
import pandas as pd
import glob

print('='*80)
print('MICROGUARD — COMPLETE RESULTS')
print('='*80)

# Collect all results from Drive
all_results = []
for f in sorted(glob.glob(f'{DRIVE_DIR}/results/*.json')):
    if 'smoke' in f or 'dataset' in f: continue
    with open(f) as fh:
        r = json.load(fh)
        all_results.append({
            'Phase': r.get('phase',''),
            'Model': r.get('model','').split('/')[-1],
            'Bal.Acc': f"{r['test_balanced_accuracy']:.4f}",
            'F1': f"{r['test_f1_macro']:.4f}",
            'Kappa': f"{r.get('test_cohen_kappa',''):.4f}" if isinstance(r.get('test_cohen_kappa'), float) else 'N/A',
            'Latency': f"{r.get('per_sample_latency_ms','N/A')}" if r.get('per_sample_latency_ms') else 'N/A',
            'Train Time': r.get('training_time_formatted','N/A'),
        })

if all_results:
    df = pd.DataFrame(all_results)
    print(df.to_string(index=False))
else:
    print('No results found.')

# Zip everything for download
import shutil
shutil.make_archive(f'{DRIVE_DIR}/microguard_all_results', 'zip', DRIVE_DIR, 'results')
print(f'\nAll results zipped: {DRIVE_DIR}/microguard_all_results.zip')
print('Download from Google Drive and share with Claude Code to update the paper!')
